# Init

In [1]:
%matplotlib qt
import numpy as np
import scipy.constants as phy_const
import matplotlib.pyplot as plt
import pickle

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import os
import re
import pandas as pd
import pickle
import glob
import sys
import configparser
from tqdm import tqdm

from cycler import cycler
import numpy as np
from scipy.ndimage import gaussian_filter1d

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.pyplot import cm
from pylab import arange, pi, sin, cos, sqrt
from matplotlib import rcParams
import matplotlib.animation as animation
from scipy import constants as cons
from matplotlib import gridspec
import matplotlib.colors as mpl_colors
from matplotlib.widgets import Slider, TextBox, Button, RadioButtons
from matplotlib import ticker
from matplotlib.path import Path
from mpl_toolkits.axes_grid1.inset_locator import mark_inset
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.ticker import FormatStrFormatter
from matplotlib.colors import Normalize, BoundaryNorm, LogNorm
from matplotlib.ticker import MaxNLocator
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.colors import ListedColormap
from matplotlib.patches import FancyArrowPatch
import matplotlib.patches as patches
import matplotlib.lines as mlines


def find_nearest(array, value):
    array = np.asarray(array)
    idx = (np.abs(array - value)).argmin()
    return idx

# Main style
plt.style.use('classic')

# Alejandro parameters
plt.rcParams["font.family"]         = 'Times New Roman'
plt.rcParams["font.weight"]         = 'normal'
plt.rcParams['figure.facecolor']    = 'white' 
plt.rcParams["font.size"]           = 12
plt.rcParams["lines.linewidth"]     = 2

# Grid parameters
plt.rcParams['axes.grid'] = True          
plt.rcParams['grid.color'] = '0.85'       
plt.rcParams['grid.linestyle'] = '-'     
plt.rcParams['grid.linewidth'] = 0.7      
plt.rcParams['grid.alpha'] = 0.7    
plt.rcParams['axes.grid.axis'] = 'both' 
plt.rcParams['axes.grid.which'] = 'major'
plt.rcParams['axes.axisbelow'] = True

# Choices of colors cycler
dashes = [( ), (5, 3), (2, 2), (6, 2, 2, 2)]  # solid, dashed, dotted, dash-dot
plt.rcParams['axes.prop_cycle'] = cycler('color', ['k', 'r', 'b', 'g'])# + cycler('ls', ['-', '--', ':', '-.']) + cycler('dashes', dashes)

# Ticks limits
plt.rcParams.update({
    'axes.autolimit_mode': 'round_numbers',  # keeps tick limits tidy
    'axes.xmargin': 0.0,  # no extra margin added
    'axes.ymargin': 0.0,
})

# Legend
plt.rcParams.update({
    'legend.loc': 'best',            # Auto place; or 'upper right', etc.
    'legend.frameon': False,         # No frame
    'legend.fontsize': 12,            # Smaller font size
    'legend.borderaxespad': 0.5,     # Padding between legend and axes
    'legend.labelspacing': 0.01,      # Vertical space between entries
    'legend.handletextpad': 0.3,     # Space between line and text
    'legend.columnspacing': 1.0,     # Horizontal space between columns
    'legend.numpoints': 1,           # One point per line symbol
    'legend.fancybox': False,        # No rounded box
    'legend.handlelength': 1,  # length of the legend line
    'legend.handleheight': 0.7,  # height of the legend handle (marker size)
})

plt.rcParams.update({
    'savefig.dpi': 300,
})


qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


# Load config

In [5]:
folder = './Results/voltage_sweep2p5mgs/'
# folder = './Results/voltage_sweep2p5mgs_modifRLC/'
files = glob.glob(folder + "test_*V/")
files_sorted = sorted(files, key=lambda x: int(re.search(r'test_(\d+)V', x).group(1)))

configFiles = [f + 'Configuration.cfg' for f in files]
cases_name = ['312.5V', '1000V']

all_configs = []
for cfg_path in configFiles:
    print(f"\nLoading configuration: {cfg_path}")

    config = configparser.ConfigParser()
    config.read(cfg_path)

    # --------------------------------------------------
    # Physical Parameters
    # --------------------------------------------------
    physicalParameters = config["Physical Parameters"]

    VG       = float(physicalParameters["Gas velocity"])
    M        = float(physicalParameters["Ion Mass"]) * phy_const.m_u
    m        = phy_const.m_e
    R1       = float(physicalParameters["Inner radius"])
    R2       = float(physicalParameters["Outer radius"])
    A0       = np.pi * (R2**2 - R1**2)
    LENGTH   = float(physicalParameters["Length of axis"])
    L0       = float(physicalParameters["Length of thruster"])
    alpha_B1 = float(physicalParameters["Anomalous transport alpha_B1"])
    alpha_B2 = float(physicalParameters["Anomalous transport alpha_B2"])
    mdot     = float(physicalParameters["Mass flow"])
    Te_Cath  = float(physicalParameters["Temperature Cathode"])
    NI0      = float(physicalParameters["Initial plasma density"])
    TE0      = float(physicalParameters["Initial Temperature"])
    Rext     = float(physicalParameters["Ballast resistor"])
    V        = float(physicalParameters["Voltage"])
    Circuit  = config.getboolean("Physical Parameters", "Circuit", fallback=False)
    Estar    = float(physicalParameters["Crossover energy"])

    # --------------------------------------------------
    # Magnetic field configuration
    # --------------------------------------------------
    MagneticFieldConfig = config["Magnetic field configuration"]
    Btype = MagneticFieldConfig["Type"]

    if Btype == "Default":
        Bmax = float(MagneticFieldConfig["Max B-field"])
        LB1  = float(MagneticFieldConfig["Length B-field 1"])
        LB2  = float(MagneticFieldConfig["Length B-field 2"])
        saveBField = config.getboolean("Magnetic field configuration", "Save B-field", fallback=False)

        B_params = dict(Bmax=Bmax, LB1=LB1, LB2=LB2, save=saveBField)

    elif Btype == "StationaryCodeBField":
        Bmax    = float(MagneticFieldConfig["Max B-field"])
        CmagIn  = float(MagneticFieldConfig["Cmag In"])
        CmagOut = float(MagneticFieldConfig["Cmag Out"])
        saveBField = config.getboolean("Magnetic field configuration", "Save B-field", fallback=False)

        B_params = dict(Bmax=Bmax, CmagIn=CmagIn, CmagOut=CmagOut, save=saveBField)

    # --------------------------------------------------
    # Numerical Parameters
    # --------------------------------------------------
    NumericsConfig = config["Numerical Parameteres"]

    NBPOINTS   = int(NumericsConfig["Number of points"])
    SAVERATE   = int(NumericsConfig["Save rate"])
    CFL        = float(NumericsConfig["CFL"])
    TIMEFINAL  = float(NumericsConfig["Final time"])
    Results    = NumericsConfig["Result dir"]
    TIMESCHEME = NumericsConfig["Time integration"]

    # --------------------------------------------------
    # Save all parameters for this config
    # --------------------------------------------------
    all_configs.append(dict(
        file=cfg_path,
        VG=VG, M=M, m=m,
        R1=R1, R2=R2, A0=A0,
        LENGTH=LENGTH, L0=L0,
        alpha_B1=alpha_B1, alpha_B2=alpha_B2,
        mdot=mdot, Te_Cath=Te_Cath,
        NI0=NI0, TE0=TE0,
        Rext=Rext, V=V, Circuit=Circuit,
        Estar=Estar,
        MagneticType=Btype,
        Magnetic=B_params,
        NBPOINTS=NBPOINTS, SAVERATE=SAVERATE,
        CFL=CFL, TIMEFINAL=TIMEFINAL,
        Results=Results, TIMESCHEME=TIMESCHEME
    ))

# ------------------------------------------------------
# All configuration blocks are now stored in all_configs
# Example use:
# ------------------------------------------------------
for cfg in all_configs:
    print("\nSimulation loaded:", cfg["file"])
    print("  Voltage:", cfg["V"])
    print("  Bmax:", cfg["Magnetic"]["Bmax"])



Loading configuration: ./Results/voltage_sweep2p5mgs/test_800V/Configuration.cfg

Loading configuration: ./Results/voltage_sweep2p5mgs/test_450V/Configuration.cfg

Loading configuration: ./Results/voltage_sweep2p5mgs/test_950V/Configuration.cfg

Loading configuration: ./Results/voltage_sweep2p5mgs/test_850V/Configuration.cfg

Loading configuration: ./Results/voltage_sweep2p5mgs/test_250V/Configuration.cfg

Loading configuration: ./Results/voltage_sweep2p5mgs/test_550V/Configuration.cfg

Loading configuration: ./Results/voltage_sweep2p5mgs/test_350V/Configuration.cfg

Loading configuration: ./Results/voltage_sweep2p5mgs/test_750V/Configuration.cfg

Loading configuration: ./Results/voltage_sweep2p5mgs/test_400V/Configuration.cfg

Loading configuration: ./Results/voltage_sweep2p5mgs/test_900V/Configuration.cfg

Loading configuration: ./Results/voltage_sweep2p5mgs/test_650V/Configuration.cfg

Loading configuration: ./Results/voltage_sweep2p5mgs/test_500V/Configuration.cfg

Simulation load

# Load pickle

In [6]:
my_dics = []

for local_file in files_sorted:
    result_file = sorted(glob.glob(local_file + "Data/*.pkl"), key=os.path.getmtime)
    data = {k: [] for k in
        ["time","ng","n1", "n02", "n12", "u1", "u02", "u12", "Te","ve","P_inlet","P_outlet",
         "Current","Voltage","MagneticB","x_center"]}
    print(local_file)
    for fpath in tqdm(result_file):
        t, P, U, Pin, Pout, J, V, B, xc = pickle.load(open(fpath, "rb"))

        data["time"].append(t)
        data["ng"].append(P[0]);  data["n1"].append(P[1]); data["n02"].append(P[2]); data["n12"].append(P[3])
        data["u1"].append(P[4]);  data["u02"].append(P[5]); data["u12"].append(P[6]); data["Te"].append(P[7])
        data["ve"].append(P[8])

        data["P_inlet"].append(Pin)
        data["P_outlet"].append(Pout)
        data["Current"].append(J)
        data["Voltage"].append(V)
        data["MagneticB"].append(B)
        data["x_center"].append(xc)

    # convert to numpy arrays
    my_dic = {k: np.array(v) for k, v in data.items()}
    my_dics.append(my_dic)

./Results/voltage_sweep2p5mgs/test_250V/


100%|██████████| 8133/8133 [00:04<00:00, 2008.37it/s]


./Results/voltage_sweep2p5mgs/test_350V/


100%|██████████| 7036/7036 [00:03<00:00, 2103.90it/s]


./Results/voltage_sweep2p5mgs/test_400V/


...

100%|██████████| 14862/14862 [00:06<00:00, 2166.89it/s]


./Results/voltage_sweep2p5mgs/test_900V/


100%|██████████| 16237/16237 [00:07<00:00, 2156.61it/s]


./Results/voltage_sweep2p5mgs/test_950V/


100%|██████████| 17766/17766 [00:07<00:00, 2222.79it/s]


# Current evolution

In [14]:
fig = plt.figure(figsize=(3.75, 2.5), dpi=250)
gs1 = gridspec.GridSpec(1, 2)
gs1.update(hspace=0.2, wspace=0.2, left=0.15, right=0.95, bottom=0.2, top=0.95)

ax1 = plt.subplot(gs1[0, 0])
ax1.set_ylabel(r'Current (A)', fontsize=12)
ax1.set_xlabel(r'$t$ (µs)', fontsize=12, labelpad=3.25)
ax1.set_xlim(0, 5000)
ax1.set_ylim(0, 20)
ax1.tick_params(axis='both', labelsize=10)
ax1.yaxis.set_major_locator(MaxNLocator(nbins=4))
ax1.xaxis.set_major_locator(MaxNLocator(nbins=4))

ax2 = plt.subplot(gs1[0, 1])
ax2.set_yticklabels('')
ax2.yaxis.set_label_position("right")
ax2.yaxis.tick_right()
ax2.set_xlabel(r'$V_\mathrm{d}$ (V)', fontsize=12, labelpad=3.25)
ax2.set_xlim(0, 1200)
ax2.set_ylim(0, 10)
ax2.tick_params(axis='both', labelsize=10)
ax2.yaxis.set_major_locator(MaxNLocator(nbins=4))
ax2.xaxis.set_major_locator(MaxNLocator(nbins=4))

from scipy.signal import find_peaks
colors = cm.plasma(np.linspace(0, 1, len(my_dics)))

for jj, my_dic in enumerate(my_dics):
    time = my_dic["time"]*1e6  # in ms
    current = gaussian_filter1d(my_dic["Current"], 5)
    Phi = my_dic["Voltage"]

    if(jj<=6):
        print(f"Processing dataset {jj} with mean voltage {Phi.mean():.2f} V")
        ax1.plot(time, current, color=colors[jj], label=Phi.mean())

    idx_ti = find_nearest(time, 3000)
    idx_te = find_nearest(time, 5000)
    Phi_mean = Phi[idx_ti:idx_te].mean(axis=0)
    current_mean = current[idx_ti:idx_te].mean(axis=0)
    current_std = current[idx_ti:idx_te].std(axis=0)
    ax2.errorbar(Phi_mean, current_mean, yerr=current_std, fmt='o', color=colors[jj], ms=2.5, capsize=3)

    # inverted = -current
    # minima, _ = find_peaks(inverted, prominence=np.std(current))
    # idx_ti = minima[0]
    # idx_te = minima[-2]
    # Phi_mean = Phi[idx_ti:idx_te].mean(axis=0)
    # current_mean = current[idx_ti:idx_te].mean(axis=0)
    # current_std = current[idx_ti:idx_te].std(axis=0)
    # ax2.errorbar(Phi_mean, current_mean, yerr=current_std, fmt='o', color=colors[jj], capsize=3)
    # ax1.scatter(time[idx_ti], current[idx_ti], color=colors[jj], marker='o')
    # ax1.scatter(time[idx_te], current[idx_te], color=colors[jj], marker='o')
# ax1.legend(fontsize=10, loc='upper right', frameon=False)

Processing dataset 0 with mean voltage 250.00 V
Processing dataset 1 with mean voltage 350.00 V
Processing dataset 2 with mean voltage 400.00 V
Processing dataset 3 with mean voltage 450.00 V
Processing dataset 4 with mean voltage 500.00 V
Processing dataset 5 with mean voltage 550.00 V
Processing dataset 6 with mean voltage 650.00 V


# Fraction evolution

## Current

In [15]:
fig = plt.figure(figsize=(3.75, 2.5), dpi=250)
gs1 = gridspec.GridSpec(1, 2)
gs1.update(hspace=0.2, wspace=0.2, left=0.15, right=0.95, bottom=0.2, top=0.95)

ax1 = plt.subplot(gs1[0, 0])
ax1.set_ylabel(r'Current (A)', fontsize=12)
ax1.set_xlabel(r'$t$ (µs)', fontsize=12, labelpad=3.25)
ax1.set_xlim(1800, 2000)
ax1.set_ylim(0, 30)
ax1.tick_params(axis='both', labelsize=10)
ax1.yaxis.set_major_locator(MaxNLocator(nbins=4))
ax1.xaxis.set_major_locator(MaxNLocator(nbins=4))

ax2 = plt.subplot(gs1[0, 1])
ax2.set_yticklabels('')
ax2.yaxis.set_label_position("right")
ax2.yaxis.tick_right()
ax2.set_xlabel(r'$V_\mathrm{d}$ (V)', fontsize=12, labelpad=3.25)
ax2.set_xlim(0, 1200)
ax2.set_ylim(0, 30)
ax2.tick_params(axis='both', labelsize=10)
ax2.yaxis.set_major_locator(MaxNLocator(nbins=4))
ax2.xaxis.set_major_locator(MaxNLocator(nbins=4))

from scipy.signal import find_peaks
colors = cm.plasma(np.linspace(0, 1, len(my_dics)))

for jj, my_dic in enumerate(my_dics):
    if(jj>=0):
        time = my_dic["time"]*1e6  # in ms
        j02 = 2*my_dic["n02"]*my_dic["u02"]
        j12 = 2*my_dic["n12"]*my_dic["u12"]
        j2 = j02 + j12
        j1 = my_dic["n1"]*my_dic["u1"]
        jion = j02 + j12 + j1
        frac02 = j02/jion*100
        frac12 = j12/jion*100

        frac02 = frac02[:, -1]
        frac12 = frac12[:, -1]
        Phi = my_dic["Voltage"]
        if(jj<=3):
            ax1.plot(time, frac02, color=colors[jj], label=Phi.mean())

        idx_ti = find_nearest(time, 1800)
        idx_te = find_nearest(time, 5000)
        Phi_mean = Phi[idx_ti:idx_te].mean(axis=0)
        frac02_mean = frac02[idx_ti:idx_te].mean(axis=0)
        frac02_std = frac02[idx_ti:idx_te].std(axis=0)
        frac12_mean = frac12[idx_ti:idx_te].mean(axis=0)
        frac12_std = frac12[idx_ti:idx_te].std(axis=0)
        ax2.errorbar(Phi_mean, frac02_mean, yerr=frac02_std, fmt='d', color=colors[jj], ms=2.5, capsize=3)
        ax2.errorbar(Phi_mean, frac12_mean, yerr=frac12_std, fmt='o', color=colors[jj], ms=2.5, capsize=3)

    # inverted = -current
    # minima, _ = find_peaks(inverted, prominence=np.std(current))
    # idx_ti = minima[0]
    # idx_te = minima[-2]
    # Phi_mean = Phi[idx_ti:idx_te].mean(axis=0)
    # current_mean = current[idx_ti:idx_te].mean(axis=0)
    # current_std = current[idx_ti:idx_te].std(axis=0)
    # ax2.errorbar(Phi_mean, current_mean, yerr=current_std, fmt='o', color=colors[jj], capsize=3)
    # ax1.scatter(time[idx_ti], current[idx_ti], color=colors[jj], marker='o')
    # ax1.scatter(time[idx_te], current[idx_te], color=colors[jj], marker='o')
# ax1.legend(fontsize=10, loc='upper right', frameon=False)

/tmp/ipykernel_24391/284641161.py:36: RuntimeWarning: invalid value encountered in divide
  frac02 = j02/jion*100
/tmp/ipykernel_24391/284641161.py:37: RuntimeWarning: invalid value encountered in divide
  frac12 = j12/jion*100


## Density

In [16]:
fig = plt.figure(figsize=(3.75, 2.5), dpi=250)
gs1 = gridspec.GridSpec(1, 2)
gs1.update(hspace=0.2, wspace=0.2, left=0.15, right=0.95, bottom=0.2, top=0.95)

ax1 = plt.subplot(gs1[0, 0])
ax1.set_ylabel(r'Current (A)', fontsize=12)
ax1.set_xlabel(r'$t$ (µs)', fontsize=12, labelpad=3.25)
ax1.set_xlim(1800, 2000)
ax1.set_ylim(0, 30)
ax1.tick_params(axis='both', labelsize=10)
ax1.yaxis.set_major_locator(MaxNLocator(nbins=4))
ax1.xaxis.set_major_locator(MaxNLocator(nbins=4))

ax2 = plt.subplot(gs1[0, 1])
ax2.set_yticklabels('')
ax2.yaxis.set_label_position("right")
ax2.yaxis.tick_right()
ax2.set_xlabel(r'$V_\mathrm{d}$ (V)', fontsize=12, labelpad=3.25)
ax2.set_xlim(0, 1200)
ax2.set_ylim(0, 30)
ax2.tick_params(axis='both', labelsize=10)
ax2.yaxis.set_major_locator(MaxNLocator(nbins=4))
ax2.xaxis.set_major_locator(MaxNLocator(nbins=4))

from scipy.signal import find_peaks
colors = cm.plasma(np.linspace(0, 1, len(my_dics)))

for jj, my_dic in enumerate(my_dics):
    if(jj>=0):
        time = my_dic["time"]*1e6  # in ms
        n02 = my_dic["n02"]
        n12 = my_dic["n12"]
        n2 = n02 + n12
        n1 = my_dic["n1"]
        nion = n02 + n12 + n1
        frac02 = n02/nion*100
        frac12 = n12/nion*100

        frac02 = frac02[:, -1]
        frac12 = frac12[:, -1]
        Phi = my_dic["Voltage"]
        ax1.plot(time, frac02, color=colors[jj], label=Phi.mean())

        idx_ti = find_nearest(time, 1800)
        idx_te = find_nearest(time, 5000)
        Phi_mean = Phi[idx_ti:idx_te].mean(axis=0)
        frac02_mean = frac02[idx_ti:idx_te].mean(axis=0)
        frac02_std = frac02[idx_ti:idx_te].std(axis=0)
        frac12_mean = frac12[idx_ti:idx_te].mean(axis=0)
        frac12_std = frac12[idx_ti:idx_te].std(axis=0)
        ax2.errorbar(Phi_mean, frac02_mean, yerr=frac02_std, fmt='d', color=colors[jj], ms=2.5, capsize=3)
        ax2.errorbar(Phi_mean, frac12_mean, yerr=frac12_std, fmt='o', color=colors[jj], ms=2.5, capsize=3)

    # inverted = -current
    # minima, _ = find_peaks(inverted, prominence=np.std(current))
    # idx_ti = minima[0]
    # idx_te = minima[-2]
    # Phi_mean = Phi[idx_ti:idx_te].mean(axis=0)
    # current_mean = current[idx_ti:idx_te].mean(axis=0)
    # current_std = current[idx_ti:idx_te].std(axis=0)
    # ax2.errorbar(Phi_mean, current_mean, yerr=current_std, fmt='o', color=colors[jj], capsize=3)
    # ax1.scatter(time[idx_ti], current[idx_ti], color=colors[jj], marker='o')
    # ax1.scatter(time[idx_te], current[idx_te], color=colors[jj], marker='o')
# ax1.legend(fontsize=10, loc='upper right', frameon=False)

# Current fraction evolution

In [17]:
fig = plt.figure(figsize=(3.5, 2.5), dpi=250)
gs1 = gridspec.GridSpec(1, 1)
gs1.update(hspace=0.2, wspace=0.1, left=0.12, right=0.95, bottom=0.18, top=0.95)

ax1 = plt.subplot(gs1[0, 0])
ax1.set_ylabel(r'Current (A)', fontsize=12)
ax1.set_xlabel(r'$t$ (µs)', fontsize=12, labelpad=3.25)
ax1.set_xlim(0, 2000)
ax1.set_ylim(0, 500)
ax1.tick_params(axis='both', labelsize=10)
ax1.yaxis.set_major_locator(MaxNLocator(nbins=4))
ax1.xaxis.set_major_locator(MaxNLocator(nbins=4))

colors = cm.plasma(np.linspace(0, 1, len(my_dics)))

for jj, my_dic in enumerate(my_dics):
    time = my_dic["time"]*1e6  # in ms
    j02 = 2*my_dic["n02"]*my_dic["u02"]
    j12 = 2*my_dic["n12"]*my_dic["u12"]
    j2 = j02 + j12
    j1 = my_dic["n1"]*my_dic["u1"]
    jion = j02 + j12 + j1

    frac2 = j2/jion*100

    idx_x1 = find_nearest(time, 2)
    idx_x2 = find_nearest(time, 2.5)

    array = frac2[:, idx_x1:idx_x2].mean(axis=1)

    ax1.plot(time, array, color=colors[jj])

ax1.legend(fontsize=10, loc='upper right', frameon=False)

/tmp/ipykernel_24391/39223168.py:24: RuntimeWarning: invalid value encountered in divide
  frac2 = j2/jion*100
/tmp/ipykernel_24391/39223168.py:33: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax1.legend(fontsize=10, loc='upper right', frameon=False)


## Time averaged density fraction

In [18]:
fig = plt.figure(figsize=(3.5, 2.5), dpi=250)
gs1 = gridspec.GridSpec(1, 1)
gs1.update(hspace=0.2, wspace=0.1, left=0.18, right=0.95, bottom=0.18, top=0.95)

ax1 = plt.subplot(gs1[0, 0])
ax1.set_ylabel(r'$n_\mathrm{2}$/$n_\mathrm{ion}$ (%)', fontsize=12)
ax1.set_xlabel(r'$x$ (cm)', fontsize=12, labelpad=3.25)
ax1.set_xlim(0, 2.5)
ax1.set_ylim(0, 20)
ax1.tick_params(axis='both', labelsize=10)
ax1.yaxis.set_major_locator(MaxNLocator(nbins=4))
ax1.xaxis.set_major_locator(MaxNLocator(nbins=4))

tis = 52*np.ones(len(my_dics))
tes = 135*np.ones(len(my_dics))

colors = cm.plasma(np.linspace(0, 1, len(my_dics)))

for jj, my_dic in enumerate(my_dics):

    time = my_dic["time"]*1e6  # in micros
    length = my_dic["x_center"][0]*100  # in cm
    idx_ti = find_nearest(time, tis[jj])
    idx_te = find_nearest(time, tes[jj])

    n02 = my_dic["n02"]
    n12 = my_dic["n12"]
    n1 = my_dic["n1"]
    n2 = n02 + n12
    nion = n02 + n12 + n1

    frac = (n2/nion)[idx_ti:idx_te].mean(axis=0)*100

    Phi_actual = 312.5174603
    Phi = my_dic['Voltage'][idx_ti:idx_te].mean()
    print(Phi)

    ax1.plot(length, frac, color=colors[jj])
    
legend = ax1.legend(frameon=True, fontsize=10, ncol=4, loc='upper left')


250.00111246839862
349.9993393119
399.9973247196951
449.9974475337906
500.0030019052312
549.9990175046978
650.0035615592634
749.9884323992819
800.0045979268647
849.9951292692621
900.000782832227
949.999962515365


/tmp/ipykernel_24391/374360655.py:40: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  legend = ax1.legend(frameon=True, fontsize=10, ncol=4, loc='upper left')


## Time averaged current fraction

In [19]:
fig = plt.figure(figsize=(3.5, 2.5), dpi=250)
gs1 = gridspec.GridSpec(1, 1)
gs1.update(hspace=0.2, wspace=0.1, left=0.18, right=0.95, bottom=0.18, top=0.95)

ax1 = plt.subplot(gs1[0, 0])
ax1.set_ylabel(r'$n_\mathrm{1}$/$n_\mathrm{ion}$ (%)', fontsize=12)
ax1.set_xlabel(r'$x$ (cm)', fontsize=12, labelpad=3.25)
ax1.set_xlim(0, 2.5)
ax1.set_ylim(0, 50)
ax1.tick_params(axis='both', labelsize=10)
ax1.yaxis.set_major_locator(MaxNLocator(nbins=4))
ax1.xaxis.set_major_locator(MaxNLocator(nbins=4))

tis = 52*np.ones(len(my_dics))
tes = 135*np.ones(len(my_dics))
colors = cm.plasma(np.linspace(0, 1, len(my_dics)))

for jj, my_dic in enumerate(my_dics):

    time = my_dic["time"]*1e6  # in micros
    length = my_dic["x_center"][0]*100  # in cm
    idx_ti = find_nearest(time, tis[jj])
    idx_te = find_nearest(time, tes[jj])

    j02 = 2*my_dic["n02"]*my_dic["u02"]
    j12 = 2*my_dic["n12"]*my_dic["u12"]
    j1 = my_dic["n1"]*my_dic["u1"]
    j2 = j02 + j12
    jion = j02 + j12 + j1

    frac = (j2/jion)[idx_ti:idx_te].mean(axis=0)*100

    Phi_actual = 312.5174603
    Phi = my_dic['Voltage'][idx_ti:idx_te].mean()
    print(Phi)

    ax1.plot(length, gaussian_filter1d(frac,1), color=colors[jj])
    
legend = ax1.legend(frameon=True, fontsize=10, ncol=4, loc='upper left')


250.00111246839862
349.9993393119
399.9973247196951
449.9974475337906
500.0030019052312
549.9990175046978
650.0035615592634
749.9884323992819
800.0045979268647
849.9951292692621
900.000782832227
949.999962515365


/tmp/ipykernel_24391/1188822805.py:31: RuntimeWarning: invalid value encountered in divide
  frac = (j2/jion)[idx_ti:idx_te].mean(axis=0)*100
/tmp/ipykernel_24391/1188822805.py:39: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  legend = ax1.legend(frameon=True, fontsize=10, ncol=4, loc='upper left')


# Time evolution with voltage

In [104]:
fig = plt.figure(figsize=(3.5, 2.5), dpi=250)
gs1 = gridspec.GridSpec(1, 1)
gs1.update(hspace=0.2, wspace=0.1, left=0.18, right=0.95, bottom=0.18, top=0.95)

ax1 = plt.subplot(gs1[0, 0])
ax1.set_ylabel(r'$j_\mathrm{2}$/$j_\mathrm{ion}$ (%)', fontsize=12)
ax1.set_xlabel(r'$V_\mathrm{d}$ (V)', fontsize=12, labelpad=3.25)
ax1.set_xlim(0, 1200)
ax1.set_ylim(0, 30)
ax1.tick_params(axis='both', labelsize=10)
ax1.yaxis.set_major_locator(MaxNLocator(nbins=4))
ax1.xaxis.set_major_locator(MaxNLocator(nbins=4))

tis = 52*np.ones(len(my_dics))
tes = 135*np.ones(len(my_dics))
colors = cm.plasma(np.linspace(0, 1, len(my_dics)))

volts = []
fracs_mean = []
frac02s_mean = []
frac12s_mean = []
fracs_std = []
frac02s_std = []
frac12s_std = []
for jj, my_dic in enumerate(my_dics):

    time = my_dic["time"]*1e6  # in micros
    length = my_dic["x_center"][0]*100  # in cm
    idx_ti = find_nearest(time, tis[jj])
    idx_te = find_nearest(time, tes[jj])

    j02 = 2*my_dic["n02"]*my_dic["u02"]
    j12 = 2*my_dic["n12"]*my_dic["u12"]
    j1 = my_dic["n1"]*my_dic["u1"]
    j2 = j02 + j12
    jion = j02 + j12 + j1

    frac = (j2/jion)[idx_ti:idx_te,0]*100
    frac02 = (j02/jion)[idx_ti:idx_te, 0]*100
    frac12 = (j12/jion)[idx_ti:idx_te, 0]*100
    
    volts.append(my_dic['Voltage'][idx_ti:idx_te].mean())
    
    fracs_mean.append(frac.mean())
    fracs_std.append(frac.std())
    frac02s_mean.append(frac02.mean())
    frac02s_std.append(frac02.std())
    frac12s_mean.append(frac12.mean())
    frac12s_std.append(frac12.std())

ax1.errorbar(volts, fracs_mean, yerr=fracs_std, fmt='o', color='k', ecolor='gray', elinewidth=2, capsize=5)
ax1.errorbar(volts, frac02s_mean, yerr=frac02s_std, fmt='o', color='#d55e00', ecolor='#d55e00', elinewidth=2, capsize=5)
ax1.errorbar(volts, frac12s_mean, yerr=frac12s_std, fmt='o', color='#009e73', ecolor='#009e73', elinewidth=2, capsize=5)

/tmp/ipykernel_192970/3527698757.py:38: RuntimeWarning: invalid value encountered in divide
  frac = (j2/jion)[idx_ti:idx_te,0]*100
/tmp/ipykernel_192970/3527698757.py:39: RuntimeWarning: invalid value encountered in divide
  frac02 = (j02/jion)[idx_ti:idx_te, 0]*100
/tmp/ipykernel_192970/3527698757.py:40: RuntimeWarning: invalid value encountered in divide
  frac12 = (j12/jion)[idx_ti:idx_te, 0]*100


<ErrorbarContainer object of 3 artists>